<a href="https://colab.research.google.com/github/lawrennd/qig-code/blob/main/examples/qutrit_gibbs_lock_clock_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Qutrit Gibbs-Lock Experiments: $\Pi_{\text{marg}}$, Loewner Kernel, and Hamiltonian Clock

This notebook implements **CIP-000F**: five experiments for the $d=3$ qutrit system that concretely verify the Hamiltonian clock construction derived in  
**"Gibbs-Lock and the Emergence of Hamiltonian Structure in the Inaccessible Game"** (Lawrence-hamiltonian26).

## Physical system

The running example throughout the paper is a single qutrit described by the Hamiltonian

$$H_\delta = \mathrm{diag}(0,\,0,\,\delta), \qquad K_0 = \beta_0 H_\delta,$$

with Gibbs state

$$\rho_0 = \frac{1}{Z}\,\mathrm{diag}(1,\,1,\,e^{-\beta_0\delta}), \qquad Z = 2 + e^{-\beta_0\delta}.$$

The spectrum has **one degenerate block** $\{\lambda_0 = \lambda_1 = 1/Z\}$ and **one separated level** $\{\lambda_2 = e^{-\beta_0\delta}/Z\}$, yielding two qualitatively different off-diagonal mode types:

| Mode type | Indices | Bohr gap | Analytical Loewner weight |
|-----------|---------|----------|---------------------------|
| In-block (degen.) | $(0,1)$, $(1,0)$ | $0$ | $1/Z$ |
| Cross-block | $(0,2)$, $(1,2)$, c.c. | $\delta$ | $(1-e^{-\beta_0\delta})/(Z\beta_0\delta)$ |

**Note on bipartite structure.** Experiments 1 and 2 require the marginal entropy constraint, which needs a bipartite partition. Those experiments use `near_bell_gibbs_frame(d=3)` (a $3\times 3 = 9$-dimensional joint system). Experiments 3, 4 and 5 operate on the single-qutrit 3×3 system described above, which is where the paper's closed forms hold exactly.

## Experiments

| # | Title | Key claim |
|---|-------|-----------|
| 3 | Loewner kernel two-sector structure | $C_{01}=1/Z$; $C_{02}=(1-e^{-\beta_0\delta})/(Z\beta_0\delta)$; smooth $\delta\to 0$ limit |
| 5 | $\hbar(\beta_0,\delta)$ closed form | $\hbar = Z^2/(2\beta_0^2\delta^2 e^{-\beta_0\delta})$; minimum near $x=\beta_0\delta\approx 2.66$ |
| 2 | Iso-marginal sector check | $\Pi_{\text{marg}}(K_0)\,R_{\text{od}} = R_{\text{od}}$ for all off-diagonal generators |
| 4 | Uniform dephasing | Decay rate $\mu_0 = c\hbar$ uniform across both mode types |
| 1 | Explicit $\Pi_{\text{marg}}$ | Rank-78 projector; complement spanned by constraint gradients |

---
## Setup

In [ ]:
# Auto-install QIG package if not available
try:
    import qig
except ImportError:
    print('Installing QIG package...')
    %pip install -q git+https://github.com/lawrennd/qig-code.git
    import qig

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

from qig.gibbs_lock import GibbsLockedFrame, infer_mu0
from qig.pair_operators import near_bell_gibbs_frame

plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'figure.dpi': 120,
})
print('qig version:', qig.__version__ if hasattr(qig, '__version__') else 'dev')

### Physical setup: the departed qutrit

We construct the single-qutrit `GibbsLockedFrame` with $H_\delta = \mathrm{diag}(0,0,\delta)$ at reference parameters $\delta=0.5$, $\beta_0=2.0$.

In [ ]:
# Reference parameters
DELTA_REF = 0.5
BETA_REF  = 2.0

def departed_qutrit_frame(delta: float, beta: float) -> GibbsLockedFrame:
    """Single-qutrit GibbsLockedFrame with H = diag(0, 0, delta)."""
    H = np.diag([0.0, 0.0, delta])
    return GibbsLockedFrame(H, beta=beta)

def Z_qutrit(delta: float, beta: float) -> float:
    """Partition function Z = 2 + exp(-beta*delta)."""
    return 2.0 + np.exp(-beta * delta)

frame_ref = departed_qutrit_frame(DELTA_REF, BETA_REF)
Z_ref = Z_qutrit(DELTA_REF, BETA_REF)

rho0 = frame_ref.rho0
vals_rho, _ = np.linalg.eigh(rho0)
vals_rho = vals_rho[::-1]  # descending

print(f'Reference: delta={DELTA_REF}, beta={BETA_REF}')
print(f'Z = {Z_ref:.6f}')
print()
print('rho_0 eigenvalues (descending):')
for i, v in enumerate(vals_rho):
    print(f'  lambda_{i} = {v:.6f}')
print()
print('Expected  lambda_0 = lambda_1 = 1/Z =', round(1/Z_ref, 6))
print('Expected  lambda_2 = exp(-beta*delta)/Z =', round(np.exp(-BETA_REF*DELTA_REF)/Z_ref, 6))
print()
print('Gibbs-lock residual ||[K0, H]||_F =', frame_ref.gibbs_lock_residual())

---
## Experiment 3 — Loewner kernel two-sector structure

**Goal.** Verify the analytical Loewner weights for both off-diagonal mode types and confirm the smooth $\delta\to 0$ degenerate limit.

The `loewner_kernel()` method returns the matrix $C$ in the eigenbasis of $\rho_0$, where $C_{ij} = k(\lambda_i, \lambda_j)$ with the BKM divided-difference kernel

$$k(p,q) = \frac{p - q}{\log p - \log q} \quad (p \ne q), \qquad k(p,p) = p.$$

The eigenvalues returned by `loewner_kernel()` are **sorted ascending**, so for our system:

| Sorted index | Paper index | Value |
|:---:|:---:|---|
| 0 | 2 | $\lambda_2 = e^{-\beta_0\delta}/Z$ (separated level) |
| 1 | 0 | $\lambda_0 = 1/Z$ (degenerate pair) |
| 2 | 1 | $\lambda_1 = 1/Z$ (degenerate pair) |

Correspondingly:
- **In-block** entry: $C[1,2] = 1/Z$
- **Cross-block** entry: $C[0,1] = C[0,2] = (1-e^{-\beta_0\delta})/(Z\beta_0\delta)$

In [ ]:
# --- Verify at the reference point ---
C, vals_sorted, vecs = frame_ref.loewner_kernel()

w_inblock_numerical  = C[1, 2].real  # in-block (degenerate pair, sorted indices 1,2)
w_cross_numerical    = C[0, 1].real  # cross-block (separated vs degenerate, sorted index 0 vs 1)

w_inblock_analytical  = 1.0 / Z_ref
w_cross_analytical    = (1 - np.exp(-BETA_REF * DELTA_REF)) / (Z_ref * BETA_REF * DELTA_REF)

print('--- Loewner weights at reference point ---')
print(f'In-block  C[1,2]:  numerical = {w_inblock_numerical:.10f}')
print(f'           analytic = {w_inblock_analytical:.10f}')
print(f'           error    = {abs(w_inblock_numerical - w_inblock_analytical):.2e}')
print()
print(f'Cross-block C[0,1]: numerical = {w_cross_numerical:.10f}')
print(f'            analytic = {w_cross_analytical:.10f}')
print(f'            error    = {abs(w_cross_numerical - w_cross_analytical):.2e}')

# Assertions
assert abs(w_inblock_numerical - w_inblock_analytical) < 1e-10, "In-block weight mismatch"
assert abs(w_cross_numerical   - w_cross_analytical)   < 1e-10, "Cross-block weight mismatch"
print()
print('✓ Both weights match analytical closed forms to 1e-10')

In [ ]:
# --- Verify over a (beta, delta) grid ---
betas  = np.linspace(0.5, 5.0, 10)
deltas = np.logspace(-2, np.log10(2.0), 20)  # log-spaced to probe delta -> 0 limit

max_err_inblock = 0.0
max_err_cross   = 0.0

for beta in betas:
    for delta in deltas:
        f = departed_qutrit_frame(delta, beta)
        C_grid, _, _ = f.loewner_kernel()
        Z = Z_qutrit(delta, beta)

        # In-block: sorted indices 1,2 are the degenerate pair
        w_in_num  = C_grid[1, 2].real
        w_in_ana  = 1.0 / Z
        max_err_inblock = max(max_err_inblock, abs(w_in_num - w_in_ana))

        # Cross-block: sorted index 0 (separated) vs sorted index 1 (degenerate)
        w_cr_num  = C_grid[0, 1].real
        w_cr_ana  = (1 - np.exp(-beta * delta)) / (Z * beta * delta)
        max_err_cross = max(max_err_cross, abs(w_cr_num - w_cr_ana))

print(f'Grid size: {len(betas)} x {len(deltas)} = {len(betas)*len(deltas)} points')
print(f'Max error in-block:   {max_err_inblock:.2e}')
print(f'Max error cross-block: {max_err_cross:.2e}')

assert max_err_inblock < 1e-10, f"In-block weight error too large: {max_err_inblock:.2e}"
assert max_err_cross   < 1e-10, f"Cross-block weight error too large: {max_err_cross:.2e}"
print()
print('✓ Both weights verified over full (beta, delta) grid to 1e-10')

In [ ]:
# --- Verify smooth delta -> 0 limit ---
# lim_{delta->0} (1 - exp(-beta*delta)) / (Z * beta * delta)  =  1/Z|_{delta=0} = 1/3
beta_check = 2.0
tiny_deltas = np.logspace(-6, -1, 30)
cross_weights  = []
inblock_weights = []

for delta in tiny_deltas:
    f = departed_qutrit_frame(delta, beta_check)
    Cg, _, _ = f.loewner_kernel()
    cross_weights.append(Cg[0, 1].real)
    inblock_weights.append(Cg[1, 2].real)

# At delta=0: Z=3, so both limits should equal 1/3
limit_expected = 1.0 / 3.0
err_limit = abs(cross_weights[0] - limit_expected)
print(f'Cross-block weight at delta={tiny_deltas[0]:.1e}: {cross_weights[0]:.10f}')
print(f'In-block  weight at delta={tiny_deltas[0]:.1e}: {inblock_weights[0]:.10f}')
print(f'Expected limit 1/3 = {limit_expected:.10f}')
print(f'Cross-block error from limit: {err_limit:.2e}')

assert err_limit < 1e-4, f"delta->0 limit not approached: error = {err_limit:.2e}"
print()
print('✓ Cross-block weight converges to in-block weight (1/Z|_{delta=0} = 1/3) as delta -> 0')

In [ ]:
# --- Plot: Loewner weights vs x = beta * delta (fixing beta) ---
beta_plot = 2.0
x_vals    = np.logspace(-2, 1, 200)  # x = beta * delta
delta_plot = x_vals / beta_plot

w_inblock_plot = []
w_cross_plot   = []
w_analytic_inblock = []
w_analytic_cross   = []

for delta in delta_plot:
    f = departed_qutrit_frame(delta, beta_plot)
    Cg, _, _ = f.loewner_kernel()
    w_inblock_plot.append(Cg[1, 2].real)
    w_cross_plot.append(Cg[0, 1].real)
    Z = Z_qutrit(delta, beta_plot)
    w_analytic_inblock.append(1.0 / Z)
    w_analytic_cross.append((1 - np.exp(-beta_plot * delta)) / (Z * beta_plot * delta))

fig, ax = plt.subplots(figsize=(6, 4))

ax.semilogx(x_vals, w_inblock_plot,       'C0-',  lw=2, label=r'In-block $C_{12}$ (numerical)')
ax.semilogx(x_vals, w_analytic_inblock,   'C0--', lw=1.5, alpha=0.6, label=r'In-block $1/Z$ (analytic)')
ax.semilogx(x_vals, w_cross_plot,         'C1-',  lw=2, label=r'Cross-block $C_{01}$ (numerical)')
ax.semilogx(x_vals, w_analytic_cross,     'C1--', lw=1.5, alpha=0.6,
            label=r'Cross-block $(1-e^{-x})/(Zx)$ (analytic)')

ax.axhline(1/3, color='gray', ls=':', lw=1)
ax.text(0.012, 1/3 + 0.004, r'$1/3\;(\delta\to 0$ limit)', color='gray', fontsize=9)

ax.set_xlabel(r'$x = \beta_0\delta$')
ax.set_ylabel('Loewner weight')
ax.set_title(r'Exp 3: Loewner kernel two-sector structure ($\beta_0=2$)')
ax.legend(fontsize=9)
ax.set_xlim(x_vals[0], x_vals[-1])
ax.set_ylim(0, 0.45)
plt.tight_layout()
plt.savefig('fig_exp3_loewner-kernel-two-sector.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: fig_exp3_loewner-kernel-two-sector.pdf')

### Exp 3 — Summary

- **In-block weight** $C_{12} = 1/Z$: constant for all $\delta$, confirmed to $10^{-10}$.
- **Cross-block weight** $C_{01} = (1-e^{-x})/(Zx)$ with $x = \beta_0\delta$: decreases monotonically in $x$, confirmed to $10^{-10}$.
- **Smooth degenerate limit**: as $\delta\to 0$, the cross-block weight converges to the in-block weight $1/Z|_{\delta=0} = 1/3$.

Gap-dependent amplitude suppression enters through the Loewner kernel when $\delta>0$, while the decay rate remains uniform in modular-generator coordinates (verified in Exp 4).

---
## Experiment 5 — $\hbar(\beta_0, \delta)$ closed form

**Goal.** Verify the closed-form expression for the effective Planck constant

$$\hbar = \frac{1}{\beta_0^2\,\mathrm{Var}_{\rho_0}(H_\delta)}, \qquad
  \mathrm{Var}_{\rho_0}(H_\delta) = \frac{2\delta^2 e^{-\beta_0\delta}}{Z^2},$$

giving

$$\boxed{\hbar = \frac{Z^2}{2\,\beta_0^2\,\delta^2\,e^{-\beta_0\delta}}}, \qquad Z = 2 + e^{-\beta_0\delta}.}$$

Scaling by $x = \beta_0\delta$, the $\beta_0$-independent shape is

$$\tilde\hbar(x) \equiv 2\beta_0^2\,\hbar = \frac{(2+e^{-x})^2}{x^2 e^{-x}}.$$

**Divergence regimes:**

| Limit | Behaviour |
|-------|-----------|
| $x\to 0$ ($\delta\to 0$ or $\beta_0\to 0$) | $\hbar\to\infty$ (levels merge, no clock) |
| $x\to\infty$ ($\beta_0\delta\to\infty$) | $\hbar\sim 4e^x/x^2\to\infty$ (thermal suppression) |

The **minimum** of $\tilde\hbar(x)$ occurs at the unique positive root of

$$\frac{d}{dx}\tilde\hbar = 0 \;\Longleftrightarrow\; 2x - 4 = (x+2)e^{-x},$$

which gives $x^* \approx 2.228$ (confirmed numerically below).

In [ ]:
# --- Exp 5: verify closed form at reference point ---
delta_e5, beta_e5 = DELTA_REF, BETA_REF

frame_e5 = departed_qutrit_frame(delta_e5, beta_e5)
H_e5   = frame_e5.H
rho_e5 = frame_e5.rho0

var_num   = np.real(np.trace(rho_e5 @ H_e5 @ H_e5)) - np.real(np.trace(rho_e5 @ H_e5))**2
hbar_num  = 1.0 / (beta_e5**2 * var_num)

Z_e5      = Z_qutrit(delta_e5, beta_e5)
var_ana   = 2 * delta_e5**2 * np.exp(-beta_e5 * delta_e5) / Z_e5**2
hbar_ana  = Z_e5**2 / (2 * beta_e5**2 * delta_e5**2 * np.exp(-beta_e5 * delta_e5))

print('--- hbar at reference point ---')
print(f'Var(H) numerical  = {var_num:.12f}')
print(f'Var(H) analytical = {var_ana:.12f}')
print(f'hbar   numerical  = {hbar_num:.12f}')
print(f'hbar   analytical = {hbar_ana:.12f}')
print(f'error             = {abs(hbar_num - hbar_ana):.2e}')

assert abs(hbar_num - hbar_ana) < 1e-10, "hbar mismatch at reference point"
print()
print('✓ hbar matches closed form to 1e-10')

In [ ]:
# --- Verify over (beta, delta) grid ---
betas_e5  = np.linspace(0.5, 5.0, 10)
deltas_e5 = np.logspace(-1.5, np.log10(3.0), 20)

max_err = 0.0
for beta in betas_e5:
    for delta in deltas_e5:
        f    = departed_qutrit_frame(delta, beta)
        H_g  = f.H
        rho_g = f.rho0
        var  = np.real(np.trace(rho_g @ H_g @ H_g)) - np.real(np.trace(rho_g @ H_g))**2
        hbar = 1.0 / (beta**2 * var)
        Z    = Z_qutrit(delta, beta)
        hbar_a = Z**2 / (2 * beta**2 * delta**2 * np.exp(-beta * delta))
        max_err = max(max_err, abs(hbar - hbar_a))

print(f'Grid size: {len(betas_e5)} x {len(deltas_e5)} = {len(betas_e5)*len(deltas_e5)} points')
print(f'Max |hbar_num - hbar_analytic| = {max_err:.2e}')
assert max_err < 1e-10, f"Grid hbar error too large: {max_err:.2e}"
print()
print('✓ Closed form verified over full (beta, delta) grid to 1e-10')

In [ ]:
# --- Find minimum and verify divergence limits ---
from scipy.optimize import brentq, minimize_scalar

def hbar_shape(x):
    """Dimensionless shape tilde_hbar(x) = (2+exp(-x))^2 / (x^2 * exp(-x))."""
    return (2 + np.exp(-x))**2 / (x**2 * np.exp(-x))

# Numerical minimum
res = minimize_scalar(hbar_shape, bounds=(0.5, 8.0), method='bounded')
x_num   = res.x
hbar_min = res.fun

# Analytical critical-point equation: d/dx[tilde_hbar] = 0
# Rearranges to 2x - 4 = (x+2)*exp(-x)
def crit_eq(x):
    return 2*x - 4 - (x+2)*np.exp(-x)

x_ana = brentq(crit_eq, 1.0, 5.0)
print(f'Numerical  minimum at x* = {x_num:.10f}')
print(f'Analytical root (2x-4=(x+2)e^{{-x}}): x* = {x_ana:.10f}')
print(f'Residual of critical-point equation at x*: {crit_eq(x_ana):.2e}')
print(f'Agreement between methods: {abs(x_num - x_ana):.2e}')

assert abs(x_num - x_ana) < 1e-6, "Numerical and analytical x* disagree"

# Verify divergence at small x (delta -> 0 limit)
x_small  = np.logspace(-3, -0.5, 5)
hbar_small = hbar_shape(x_small)
print(f'\nDivergence at small x: tilde_hbar({x_small[0]:.4f}) = {hbar_small[0]:.2e}  (expected >> 1)')
assert hbar_small[0] > 1e4, "No divergence at x -> 0"

# Verify divergence at large x (beta*delta -> inf)
x_large  = np.array([5, 10, 20, 50])
hbar_large = hbar_shape(x_large)
print(f'Divergence at large x: tilde_hbar({x_large[-1]}) = {hbar_large[-1]:.2e}  (expected >> 1)')
assert hbar_large[-1] > 1e15, "No divergence at x -> inf"

print()
print(f'✓ Minimum confirmed at x* = {x_ana:.4f}  (analytical: 2x-4=(x+2)e^{{-x}})')
print('✓ Divergence confirmed at both x → 0 and x → ∞')

In [ ]:
# --- Plot: hbar vs x = beta*delta ---
x_plot   = np.logspace(-1.5, 2.0, 400)
hbar_plot = hbar_shape(x_plot)

fig, ax = plt.subplots(figsize=(6, 4))

ax.loglog(x_plot, hbar_plot, 'C0-', lw=2, label=r'$\tilde\hbar(x) = (2+e^{-x})^2/(x^2 e^{-x})$')

# Mark minimum (using x_ana from previous cell)
ax.axvline(x_ana, color='C1', ls='--', lw=1.5, label=fr'$x^*={x_ana:.3f}$ (from $2x-4=(x+2)e^{{-x}}$)')
ax.plot(x_ana, hbar_shape(x_ana), 'C1o', ms=7)

# Asymptotic guides
x_guide = np.logspace(-1.5, -0.2, 50)
ax.loglog(x_guide, 4 / x_guide**2, 'k:', lw=1, alpha=0.5, label=r'$4/x^2$ ($x\to 0$)')
x_guide2 = np.logspace(0.5, 2.0, 50)
ax.loglog(x_guide2, 4 * np.exp(x_guide2) / x_guide2**2, 'k--', lw=1, alpha=0.5,
          label=r'$4e^x/x^2$ ($x\to\infty$)')

ax.set_xlabel(r'$x = \beta_0\delta$')
ax.set_ylabel(r'$\tilde\hbar(x)$  (units: $1/\beta_0^2$)')
ax.set_title(r'Exp 5: $\hbar(\beta_0,\delta)$ shape — two divergence regimes')
ax.legend(fontsize=9)
ax.set_xlim(x_plot[0], x_plot[-1])
ax.set_ylim(1, 1e8)
plt.tight_layout()
plt.savefig('fig_exp5_hbar-closed-form.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: fig_exp5_hbar-closed-form.pdf')

### Exp 5 — Summary

- **Closed form** $\hbar = Z^2/(2\beta_0^2\delta^2 e^{-\beta_0\delta})$ verified numerically to machine precision ($< 10^{-10}$) over a 10×20 $(\beta_0, \delta)$ grid.
- **Minimum** of $\tilde\hbar(x)$ occurs at $x^* \approx 2.228$, the unique root of $2x - 4 = (x+2)e^{-x}$ (derived analytically from $d\tilde\hbar/dx = 0$; the backlog's estimate of 2.66 was incorrect).
- **Divergence** confirmed: $\hbar\to\infty$ as $x\to 0$ (levels merge, $\sim 4/x^2$) and $\hbar\to\infty$ as $x\to\infty$ (thermally suppressed variance, $\sim 4e^x/x^2$).

The clock ticks most efficiently near $x^* \approx 2.228$ ($\beta_0\delta \approx 2.228$). Exp 4 verifies that the uniform dephasing rate $\mu_0 \propto 1/\hbar$.